# Notebook 1: Extract Raw Data & Build Manifest

**What this notebook does:**
1. Extracts all 5,301 ZIP files from the ASER dataset
2. Reads the JSON metadata inside each ZIP (child info + transcripts)
3. Builds one master CSV (`raw_manifest.csv`) with one row per audio clip
4. Prints raw data statistics and verifies against the original paper (Table 3)

**Output:** `ASER-Dataset/raw_manifest.csv` (81,423 clips, 123.72 hours)

**Paper reference:** Agarwal et al., *A Dataset for Measuring Reading Levels in India at Scale*, ICASSP 2020

## Step 1: Setup paths and configuration

In [1]:
import os
import json
import zipfile
import csv
import subprocess
from pathlib import Path
from collections import defaultdict

# Paths
PROJECT_ROOT = Path("/home/hp/Indain_children_spech")
ASER_ROOT    = PROJECT_ROOT / "ASER-Dataset"
ZIP_DIR      = ASER_ROOT / "Data"          # raw ZIP files organized by region
EXTRACT_DIR  = ASER_ROOT / "extracted"      # where we extract audio + JSON
OUTPUT_CSV   = ASER_ROOT / "raw_manifest.csv"

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

print(f"ZIP source:  {ZIP_DIR}")
print(f"Extract to:  {EXTRACT_DIR}")
print(f"Output CSV:  {OUTPUT_CSV}")

ZIP source:  /home/hp/Indain_children_spech/ASER-Dataset/Data
Extract to:  /home/hp/Indain_children_spech/ASER-Dataset/extracted
Output CSV:  /home/hp/Indain_children_spech/ASER-Dataset/raw_manifest.csv


## Step 2: Understand the raw folder structure

The raw data is organized as:
```
ASER-Dataset/Data/
  Hindi RJ/     <- Hindi, Rajasthan region
    3439.zip    <- one ZIP per child (child_id = 3439)
    2187.zip
    ...
  Hindi UP/     <- Hindi, Uttar Pradesh region
    5012.zip
    ...
  Marathi MH/   <- Marathi, Maharashtra region
    1823.zip
    ...
```

Inside each ZIP:
- `summary{child_id}.json` - child's metadata + list of reading tasks
- Multiple `.mp3` files - one audio recording per reading task

**English clips** are embedded inside Hindi and Marathi sessions (ASER tests children on both native language + English).

In [2]:
# Count ZIP files per region
for region_dir in sorted(ZIP_DIR.iterdir()):
    if region_dir.is_dir():
        zips = list(region_dir.glob("*.zip"))
        print(f"{region_dir.name}: {len(zips)} ZIP files (children)")

all_zips = sorted(ZIP_DIR.rglob("*.zip"))
print(f"\nTotal ZIP files: {len(all_zips)}")

Hindi RJ: 1863 ZIP files (children)
Hindi UP: 2096 ZIP files (children)
Marathi MH: 1342 ZIP files (children)

Total ZIP files: 5301


## Step 3: Define reading level and language detection

Each audio clip has a `que_id` like `HI_S4_P_0` which encodes:
- `HI` = Hindi session, `MR` = Marathi session
- `S4` = session number
- `P` = reading level (Paragraph)
- `0` = question number

Level codes:
- `ST` = Story, `P` = Paragraph, `WD` = Word, `L` = Letter (native language)
- `CL` = Capital Letter, `SL` = Small Letter, `W` = Word, `S` = Sentence (English)

In [3]:
# Reading level codes from que_id
LEVEL_MAP = {
    "ST": "Story",
    "P":  "Paragraph",
    "WD": "Word",
    "L":  "Letter",
    "CL": "Capital Letter (English)",
    "SL": "Small Letter (English)",
    "W":  "Word (English)",
    "S":  "Sentence (English)",
}

def get_level_and_language(que_id):
    """Extract reading level and script language from que_id like HI_S1_ST_0"""
    parts = que_id.split("_")
    if len(parts) < 3:
        return "Unknown", "Unknown"
    
    lang_code = parts[0]     # HI or MR
    level_code = parts[2]    # ST, P, WD, L, CL, SL, W, S
    
    # Determine language
    if level_code in ("CL", "SL", "W", "S"):
        language = "English"  # English tasks embedded in Hindi/Marathi sessions
    elif lang_code == "HI":
        language = "Hindi"
    elif lang_code == "MR":
        language = "Marathi"
    else:
        language = "Unknown"
    
    level = LEVEL_MAP.get(level_code, level_code)
    return level, language

# Test
print(get_level_and_language("HI_S4_P_0"))    # ('Paragraph', 'Hindi')
print(get_level_and_language("MR_S1_ST_0"))   # ('Story', 'Marathi')
print(get_level_and_language("HI_S4_CL_0"))   # ('Capital Letter (English)', 'English')

('Paragraph', 'Hindi')
('Story', 'Marathi')
('Capital Letter (English)', 'English')


## Step 4: Extract all ZIPs and build manifest

For each ZIP file:
1. Extract contents to `extracted/{region}/{child_id}/`
2. Read the `summary{child_id}.json` file
3. For each audio clip listed in the JSON, create one row in our manifest

**This takes ~5-10 minutes for all 5,301 ZIPs.**

In [4]:
records = []
errors = []

for i, zip_path in enumerate(all_zips, 1):
    if i % 500 == 0 or i == 1:
        print(f"  Processing {i}/{len(all_zips)} ...")
    
    region = zip_path.parent.name     # e.g. "Hindi RJ"
    child_id = zip_path.stem          # e.g. "3439"
    child_dir = EXTRACT_DIR / region / child_id
    child_dir.mkdir(parents=True, exist_ok=True)
    
    # Extract ZIP
    try:
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(child_dir)
    except Exception as e:
        errors.append((str(zip_path), str(e)))
        continue
    
    # Find and read the JSON file
    json_files = list(child_dir.glob("summary*.json"))
    if not json_files:
        errors.append((str(zip_path), "No summary JSON found"))
        continue
    
    try:
        with open(json_files[0], "r", encoding="utf-8") as f:
            meta = json.load(f)
    except Exception as e:
        errors.append((str(zip_path), f"JSON error: {e}"))
        continue
    
    # Extract child-level info from JSON
    age_group = meta.get("ageGroup", "Unknown")
    student_class = meta.get("studClass", "Unknown")
    native_proficiency = meta.get("nativeProficiency", "Unknown")
    english_proficiency = meta.get("englishProficiency", "Unknown")
    date_str = meta.get("date", "")
    
    # Process each audio clip in the session
    for item in meta.get("sequenceList", []):
        transcript = item.get("que_text", "").strip()
        recording_name = item.get("recordingName", "")
        que_id = item.get("que_id", "")
        is_correct = item.get("isCorrect", None)
        num_mistakes = item.get("noOfMistakes", "0")
        
        audio_path = child_dir / recording_name
        if not audio_path.exists():
            errors.append((str(zip_path), f"Missing audio: {recording_name}"))
            continue
        
        reading_level, language = get_level_and_language(que_id)
        
        records.append({
            "audio_path": str(audio_path),
            "transcript": transcript,
            "language": language,
            "region": region,
            "reading_level": reading_level,
            "que_id": que_id,
            "is_correct": is_correct,
            "num_mistakes": num_mistakes,
            "age_group": age_group,
            "student_class": student_class,
            "child_id": child_id,
            "date": date_str,
        })

print(f"\nExtraction complete!")
print(f"Total audio clips: {len(records)}")
print(f"Errors: {len(errors)}")
if errors:
    print("First 5 errors:")
    for e in errors[:5]:
        print(f"  {e}")

  Processing 1/5301 ...
  Processing 500/5301 ...
  Processing 1000/5301 ...
  Processing 1500/5301 ...
  Processing 2000/5301 ...
  Processing 2500/5301 ...
  Processing 3000/5301 ...
  Processing 3500/5301 ...
  Processing 4000/5301 ...
  Processing 4500/5301 ...
  Processing 5000/5301 ...

Extraction complete!
Total audio clips: 81423
Errors: 235
First 5 errors:
  ('/home/hp/Indain_children_spech/ASER-Dataset/Data/Hindi RJ/3633.zip', 'Missing audio: HI_S2_P_1.mp3')
  ('/home/hp/Indain_children_spech/ASER-Dataset/Data/Hindi RJ/3633.zip', 'Missing audio: HI_S2_ST_0.mp3')
  ('/home/hp/Indain_children_spech/ASER-Dataset/Data/Hindi RJ/3633.zip', 'Missing audio: HI_S2_CL_1.mp3')
  ('/home/hp/Indain_children_spech/ASER-Dataset/Data/Hindi RJ/3633.zip', 'Missing audio: HI_S2_CL_3.mp3')
  ('/home/hp/Indain_children_spech/ASER-Dataset/Data/Hindi RJ/3633.zip', 'Missing audio: HI_S2_CL_0.mp3')


## Step 5: Compute duration for every clip using ffprobe

We measure the exact duration of each audio file using ffprobe (part of ffmpeg).
This takes ~10-15 minutes for 81K files.

In [5]:
def get_duration(audio_path):
    """Get audio duration in seconds using ffprobe"""
    try:
        result = subprocess.run(
            ["ffprobe", "-v", "error",
             "-show_entries", "format=duration",
             "-of", "default=noprint_wrappers=1:nokey=1",
             audio_path],
            capture_output=True, text=True, timeout=10
        )
        return float(result.stdout.strip()) if result.stdout.strip() else 0.0
    except:
        return 0.0

print("Computing duration for all clips...")
for i, r in enumerate(records):
    r["duration_sec"] = get_duration(r["audio_path"])
    if (i + 1) % 5000 == 0:
        print(f"  {i+1}/{len(records)} done")

print(f"Done. All {len(records)} durations computed.")

Computing duration for all clips...
  5000/81423 done
  10000/81423 done
  15000/81423 done
  20000/81423 done
  25000/81423 done
  30000/81423 done
  35000/81423 done
  40000/81423 done
  45000/81423 done
  50000/81423 done
  55000/81423 done
  60000/81423 done
  65000/81423 done
  70000/81423 done
  75000/81423 done
  80000/81423 done
Done. All 81423 durations computed.


## Step 6: Save raw_manifest.csv

In [6]:
fieldnames = ["audio_path", "transcript", "language", "region", "reading_level",
              "que_id", "is_correct", "num_mistakes", "age_group",
              "student_class", "child_id", "date", "duration_sec"]

with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(records)

print(f"Saved: {OUTPUT_CSV}")
print(f"Rows: {len(records)}")

Saved: /home/hp/Indain_children_spech/ASER-Dataset/raw_manifest.csv
Rows: 81423


## Step 7: Raw data statistics

Let's see what we have and verify against the paper's Table 3.

In [7]:
total_clips = len(records)
total_hours = sum(r["duration_sec"] for r in records) / 3600
total_children = len(set(r["child_id"] for r in records))

print("=" * 60)
print("RAW DATASET OVERVIEW")
print("=" * 60)
print(f"Total clips:    {total_clips:,}")
print(f"Total duration: {total_hours:.2f} hours")
print(f"Total children: {total_children:,}")

# By region
print(f"\n--- By Region ---")
for region in ["Hindi RJ", "Hindi UP", "Marathi MH"]:
    clips = [r for r in records if r["region"] == region]
    hrs = sum(r["duration_sec"] for r in clips) / 3600
    children = len(set(r["child_id"] for r in clips))
    print(f"  {region:<15} {len(clips):>6} clips | {hrs:>6.2f} hrs | {children:>5} children")

# By language
print(f"\n--- By Language ---")
for lang in ["Hindi", "Marathi", "English"]:
    clips = [r for r in records if r["language"] == lang]
    hrs = sum(r["duration_sec"] for r in clips) / 3600
    print(f"  {lang:<15} {len(clips):>6} clips | {hrs:>6.2f} hrs")

# By reading level
print(f"\n--- By Reading Level ---")
level_data = defaultdict(lambda: {"clips": 0, "hrs": 0})
for r in records:
    level_data[r["reading_level"]]["clips"] += 1
    level_data[r["reading_level"]]["hrs"] += r["duration_sec"] / 3600
for lvl in sorted(level_data, key=lambda x: -level_data[x]["hrs"]):
    d = level_data[lvl]
    print(f"  {lvl:<30} {d['clips']:>6} clips | {d['hrs']:>6.2f} hrs")

RAW DATASET OVERVIEW
Total clips:    81,423
Total duration: 123.72 hours
Total children: 5,260

--- By Region ---
  Hindi RJ         31114 clips |  46.17 hrs |  1836 children
  Hindi UP         31509 clips |  44.09 hrs |  2086 children
  Marathi MH       18800 clips |  33.45 hrs |  1338 children

--- By Language ---
  Hindi            15880 clips |  53.84 hrs
  Marathi           4421 clips |  22.39 hrs
  English          60867 clips |  47.19 hrs

--- By Reading Level ---
  Story                            3109 clips |  43.01 hrs
  Paragraph                        4523 clips |  20.37 hrs
  Capital Letter (English)        22033 clips |  14.03 hrs
  Word (English)                  14632 clips |  13.00 hrs
  Small Letter (English)          17961 clips |  10.34 hrs
  Sentence (English)               6241 clips |   9.81 hrs
  Word                             5976 clips |   7.11 hrs
  Letter                           6773 clips |   5.92 hrs
  Cl                                175 clips |   0.

## Step 8: Verify against original paper (Table 3)

The paper reports: 5,301 subjects, 81,330 clips, 123.48 hours.
Our numbers should be very close.

In [8]:
# Paper Table 3 numbers
paper = {
    "Hindi Story":     {"clips": 2244,  "hrs": 28.30},
    "Hindi Para":      {"clips": 3290,  "hrs": 14.81},
    "Hindi Word":      {"clips": 4669,  "hrs": 5.65},
    "Hindi Letter":    {"clips": 5667,  "hrs": 4.92},
    "Marathi Story":   {"clips": 860,   "hrs": 14.68},
    "Marathi Para":    {"clips": 1225,  "hrs": 5.40},
    "Marathi Word":    {"clips": 1307,  "hrs": 1.45},
    "Marathi Letter":  {"clips": 1106,  "hrs": 0.99},
    "Eng Sentence":    {"clips": 6224,  "hrs": 9.80},
    "Eng Word":        {"clips": 14611, "hrs": 13.00},
    "Eng Cap Letter":  {"clips": 22186, "hrs": 14.14},
    "Eng Sm Letter":   {"clips": 17941, "hrs": 10.34},
}

# Our numbers for same categories
hindi = [r for r in records if r["region"] in ("Hindi RJ", "Hindi UP")]
marathi = [r for r in records if r["region"] == "Marathi MH"]

ours = {
    "Hindi Story":     [r for r in hindi if r["reading_level"] == "Story"],
    "Hindi Para":      [r for r in hindi if r["reading_level"] == "Paragraph"],
    "Hindi Word":      [r for r in hindi if r["reading_level"] == "Word"],
    "Hindi Letter":    [r for r in hindi if r["reading_level"] == "Letter"],
    "Marathi Story":   [r for r in marathi if r["reading_level"] == "Story"],
    "Marathi Para":    [r for r in marathi if r["reading_level"] == "Paragraph"],
    "Marathi Word":    [r for r in marathi if r["reading_level"] == "Word"],
    "Marathi Letter":  [r for r in marathi if r["reading_level"] == "Letter"],
    "Eng Sentence":    [r for r in records if r["reading_level"] == "Sentence (English)"],
    "Eng Word":        [r for r in records if r["reading_level"] == "Word (English)"],
    "Eng Cap Letter":  [r for r in records if r["reading_level"] == "Capital Letter (English)"],
    "Eng Sm Letter":   [r for r in records if r["reading_level"] == "Small Letter (English)"],
}

print(f"{'Category':<20} {'Paper Clips':>12} {'Our Clips':>12} {'Paper Hrs':>10} {'Our Hrs':>10} {'Match?':>8}")
print("-" * 75)
for key in paper:
    p = paper[key]
    o_clips = len(ours[key])
    o_hrs = sum(r["duration_sec"] for r in ours[key]) / 3600
    match = "YES" if abs(o_clips - p["clips"]) < 200 and abs(o_hrs - p["hrs"]) < 1.0 else "NO"
    print(f"  {key:<18} {p['clips']:>10,} {o_clips:>12,} {p['hrs']:>10.2f} {o_hrs:>10.2f} {match:>8}")

paper_total = sum(p["clips"] for p in paper.values())
paper_hrs = sum(p["hrs"] for p in paper.values())
print("-" * 75)
print(f"  {'TOTAL':<18} {paper_total:>10,} {total_clips:>12,} {paper_hrs:>10.2f} {total_hours:>10.2f}")
print(f"\nNumbers match the paper. Extraction verified.")

Category              Paper Clips    Our Clips  Paper Hrs    Our Hrs   Match?
---------------------------------------------------------------------------
  Hindi Story             2,244        2,252      28.30      28.36      YES
  Hindi Para              3,290        3,307      14.81      15.00      YES
  Hindi Word              4,669        4,589       5.65       5.50      YES
  Hindi Letter            5,667        5,697       4.92       4.94      YES
  Marathi Story             860          857      14.68      14.65      YES
  Marathi Para            1,225        1,216       5.40       5.37      YES
  Marathi Word            1,307        1,387       1.45       1.61      YES
  Marathi Letter          1,106        1,076       0.99       0.98      YES
  Eng Sentence            6,224        6,241       9.80       9.81      YES
  Eng Word               14,611       14,632      13.00      13.00      YES
  Eng Cap Letter         22,186       22,033      14.14      14.03      YES
  Eng Sm L

## Done!

**Output:** `ASER-Dataset/raw_manifest.csv` with 81,423 clips and 123.72 hours.

**Next:** Run `02_clean_audio.ipynb` to apply VAD, volume normalization, and chunk long clips.